# Analyzer
Base reproducible para estudiar la evidencia de seguridad generada por el Miner.

**Pregunta general:** ¿Cómo se distribuyen los hallazgos de seguridad en la organización y qué patrones permiten identificar prioridades de revisión?

Los hallazgos son alertas o coincidencias, no vulnerabilidades confirmadas. Los análisis fallidos no representan ausencia de problemas.

## Preguntas de investigación

1. ¿Los hallazgos se concentran en pocos repositorios?
2. ¿Qué tipos de problemas de código detectados por CodeQL se repiten entre proyectos?
3. ¿Dónde se concentran los puntajes altos, distinguiendo sus escalas y fuentes?
4. ¿Qué paquetes y vulnerabilidades de Grype comparten varios repositorios?
5. ¿Existe relación entre los conteos de CodeQL y Grype entre repositorios con cobertura completa?
6. ¿En qué archivos se concentran los hallazgos de CodeQL dentro de cada repositorio?

Cada pregunta se desarrolla mediante método, resultados, interpretación y limitaciones.
No contar cada puntaje como un hallazgo nuevo.


## Preparación de los datos

Importación de librerías, selección de la evidencia, validación y creación de tablas.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


In [ ]:
# Localizar el proyecto desde la carpeta principal o desde notebooks.
current_directory = Path.cwd()
project_candidates = [
    current_directory,
    current_directory.parent,
    current_directory / 'analyzer',
]
ANALYZER_ROOT = next(
    (path for path in project_candidates if (path / 'notebooks/analyzer.ipynb').is_file()),
    None,
)
if ANALYZER_ROOT is None:
    raise FileNotFoundError('No se encontró la carpeta analyzer.')

# Copia completa; el origen está registrado en data/django/provenance.json.
EVIDENCE_ROOT = ANALYZER_ROOT / 'data' / 'django' / 'clone-x269596i'
DATASET_PATH = EVIDENCE_ROOT / 'dataset.json'
print(f'Dataset: {DATASET_PATH}')


In [ ]:
with DATASET_PATH.open(encoding='utf-8') as dataset_file:
    dataset = json.load(dataset_file)

if not isinstance(dataset, dict):
    raise ValueError('El dataset debe ser un objeto JSON.')

supported_versions = {'1.0', '1.1'}
if dataset.get('schema_version') not in supported_versions:
    raise ValueError('Versión de dataset no compatible.')

for section in ('repositories', 'findings'):
    records = dataset.get(section)
    if not isinstance(records, list):
        raise ValueError(f'La sección {section} debe ser una lista.')
    if any(not isinstance(record, dict) for record in records):
        raise ValueError(f'La sección {section} contiene registros inválidos.')

repository_records = dataset['repositories']
finding_records = dataset['findings']
repository_names = {record['repository'] for record in repository_records}

for finding in finding_records:
    if finding.get('repository') not in repository_names:
        raise ValueError('Hay hallazgos de repositorios ausentes del dataset.')
    for field in ('tool', 'vulnerability_id'):
        if not isinstance(finding.get(field), str) or not finding[field].strip():
            raise ValueError(f'Un hallazgo no contiene un {field} válido.')

print(f"Organización: {dataset['organization']}")
print(f'Repositorios: {len(repository_records)}')
print(f'Hallazgos: {len(finding_records)}')



In [ ]:
repository_columns = [
    'repository',
    'clone_status',
    'codeql_status',
    'grype_status',
    'clone_error',
    'codeql_error',
    'grype_error',
]
finding_columns = [
    'repository',
    'tool',
    'vulnerability_id',
    'description',
    'severity',
    'severity_kind',
    'locations',
    'language',
    'package',
    'package_type',
    'version',
    'commit',
    'scores',
]
score_columns = [
    'finding_index',
    'repository',
    'tool',
    'value',
    'system',
    'source',
    'version',
    'vector',
    'vulnerability_id',
]

repositories = pd.DataFrame(repository_records).reindex(columns=repository_columns)
findings = pd.DataFrame(finding_records).reindex(columns=finding_columns)

score_rows = []
for finding_index, finding in enumerate(finding_records):
    for score in finding.get('scores', []):
        score_rows.append({
            **score,
            'finding_index': finding_index,
            'repository': finding['repository'],
            'tool': finding['tool'],
        })
scores = pd.DataFrame(score_rows).reindex(columns=score_columns)

display(repositories.head())
display(findings.head())
display(scores.head())



## Cobertura antes del análisis
Separar los estados por herramienta. `not_run`, fallos y análisis parciales deben considerarse al definir las poblaciones de comparación.

In [ ]:
status_columns = ['clone_status', 'codeql_status', 'grype_status']
repository_stages = repositories.melt(
    id_vars='repository',
    value_vars=status_columns,
    var_name='stage',
    value_name='status',
)
coverage = (
    repository_stages.groupby(['stage', 'status'])
    .size()
    .rename('repositories')
    .reset_index()
)
findings_by_tool = (
    findings.groupby('tool')
    .size()
    .rename('findings')
    .reset_index()
)

display(coverage)
display(findings_by_tool)



## Hallazgos repetidos por repositorio

Agrupamos por herramienta, identificador y repositorio. Coincidencias totales mide registros del Miner; repositorios afectados mide la extensión del hallazgo entre proyectos. Las reglas de CodeQL y los identificadores de Grype se mantienen separados. No se agrupa por puntaje ni se eliminan coincidencias de paquetes distintos.

In [ ]:
frecuencias = (
    findings.groupby(['tool', 'vulnerability_id', 'repository'])
    .size()
    .rename('coincidencias')
    .reset_index()
    .sort_values(['tool', 'vulnerability_id', 'repository'])
)

repetidos = (
    frecuencias.groupby(['tool', 'vulnerability_id'])
    .agg(
        repositorios_afectados=('repository', 'nunique'),
        coincidencias_totales=('coincidencias', 'sum'),
    )
    .reset_index()
    .sort_values(
        ['repositorios_afectados', 'coincidencias_totales', 'tool', 'vulnerability_id'],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

display(frecuencias, repetidos)

# Matriz de coincidencias: filas por hallazgo y columnas por repositorio.
matriz_hallazgos = frecuencias.pivot(
    index=['tool', 'vulnerability_id'], columns='repository', values='coincidencias'
).reindex(columns=sorted(repositories['repository'].unique())).fillna(0).astype(int)
display(matriz_hallazgos)
# Un cero solo indica ausencia de registros; consultar coverage para saber si hubo análisis.


## Pregunta 1. ¿Los hallazgos se concentran en pocos repositorios?

### Método

Contar los registros por repositorio y herramienta y calcular su proporción dentro de cada herramienta.
Incluir todos los repositorios del dataset, conservando el estado del análisis. Un conteo cero no demuestra ausencia de problemas si el análisis falló.

### Resultados


In [ ]:
conteos_por_repositorio = (
    findings.groupby(['repository', 'tool'])
    .size()
    .unstack('tool', fill_value=0)
    .reindex(
        index=repositories['repository'],
        columns=['codeql', 'grype'],
        fill_value=0,
    )
    .fillna(0)
    .astype(int)
)

concentracion = (
    conteos_por_repositorio
    .reset_index()
    .melt(
        id_vars='repository',
        var_name='tool',
        value_name='coincidencias',
    )
)
totales_por_herramienta = concentracion.groupby('tool')['coincidencias'].transform('sum')
concentracion['proporcion'] = (
    concentracion['coincidencias'] / totales_por_herramienta.replace(0, float('nan'))
)
concentracion = concentracion.sort_values(
    ['tool', 'coincidencias', 'repository'],
    ascending=[True, False, True],
)

estados_por_herramienta = repositories.melt(
    id_vars='repository',
    value_vars=['codeql_status', 'grype_status'],
    var_name='tool',
    value_name='status',
)
estados_por_herramienta['tool'] = (
    estados_por_herramienta['tool'].str.removesuffix('_status')
)
concentracion = concentracion.merge(
    estados_por_herramienta,
    on=['repository', 'tool'],
    validate='one_to_one',
)

display(concentracion)


### Interpretación

Pendiente: identificar qué repositorios acumulan la mayor proporción de hallazgos por herramienta y valorar si existe concentración.

### Limitaciones

Los conteos no están ajustados por tamaño del código ni número de dependencias. Se cuentan coincidencias, no vulnerabilidades únicas. Los fallos y la cobertura parcial condicionan la comparación.


## Pregunta 2. ¿Qué tipos de problemas de código se repiten entre proyectos?

### Método

Filtrar CodeQL y agrupar por identificador de regla. Una regla compartida aparece en al menos dos repositorios distintos. Distinguir el número de repositorios afectados del total de coincidencias: muchos casos en un solo proyecto no significan repetición entre proyectos.

### Resultados


In [ ]:
reglas_compartidas = repetidos.loc[
    (repetidos['tool'] == 'codeql')
    & (repetidos['repositorios_afectados'] >= 2)
].copy()

repositorios_por_regla = frecuencias.loc[
    (frecuencias['tool'] == 'codeql')
    & frecuencias['vulnerability_id'].isin(reglas_compartidas['vulnerability_id'])
].copy()

display(reglas_compartidas)
display(repositorios_por_regla)


### Interpretación

Pendiente: identificar las reglas compartidas y explicar qué tipos de problemas representan. Si no hay reglas presentes en varios repositorios, registrar ese resultado sin confundirlo con ausencia de hallazgos.

### Limitaciones

Compartir una regla significa compartir un tipo de alerta, no la misma vulnerabilidad concreta. La comparación depende de los lenguajes y repositorios analizados correctamente; no prueba que el problema exista ni que sea explotable.


## Pregunta 3. ¿En qué repositorios se concentran los hallazgos con puntajes altos?

### Introducción

Esta pregunta estudia la distribución de los hallazgos según los puntajes originales disponibles. Queremos distinguir los repositorios que acumulan muchos hallazgos con puntajes altos de aquellos donde estos representan una proporción importante de los hallazgos puntuados.

Las escalas de CodeQL y Grype se estudian por separado. Una puntuación alta orienta la revisión, pero no confirma la explotabilidad ni determina por sí sola el riesgo del repositorio.

### Método


#### CodeQL: puntaje de la regla

Usaremos el valor original `security-severity` que el Miner extrae de los metadatos de la regla en el SARIF. Es un valor entre **0,0 y 10,0** asociado a la regla o query que generó el hallazgo. GitHub clasifica sus valores como **Low** (0,1–3,9), **Medium** (4,0–6,9), **High** (7,0–8,9) y **Critical** (9,0–10,0). En el dataset se conserva en `scores`, con `system = "security-severity"` y `source = "codeql"`. El Analyzer utiliza `value` directamente: no consulta cada identificador ni convierte `warning`, `error` o `note` en números.

Para las consultas de seguridad incorporadas a las suites Default o Extended, GitHub documenta que el puntaje se asigna a la regla a partir del percentil 75 de los puntajes CVSS de CVE asociados a sus etiquetas CWE. Por ello, representa la severidad del tipo de problema detectado; no es una evaluación CVSS individual de cada ubicación de código ni demuestra su explotabilidad.

Fuente: [GitHub: cálculo de la severidad de seguridad](https://docs.github.com/en/code-security/concepts/code-scanning/code-scanning-alerts#calculation-of-security-severity-levels).

#### Grype: puntajes CVSS publicados

Usaremos los puntajes base que Grype proporciona en `scores`, con `system = "CVSS"`, conservando la fuente, versión, vector e identificador relacionado cuando estén disponibles. No se calcula una puntuación nueva a partir de la etiqueta cualitativa.

#### Decisiones metodológicas

- Se considera puntaje alto un valor mayor o igual a **7,0**.
- Cada hallazgo se cuenta una sola vez. Cuando contiene varios puntajes, se utiliza el máximo como valor representativo para priorización; las fuentes originales permanecen en `scores`.
- Una lista `scores` vacía significa que no hay puntuación disponible: no equivale a cero ni a severidad baja.
- CodeQL y Grype se resumen en filas separadas. Sus valores no se suman ni se promedian entre herramientas.
- La concentración se estudia mediante el número de hallazgos altos, su proporción entre los hallazgos puntuados y la cobertura de puntuación.


In [ ]:
HIGH_SCORE_THRESHOLD = 7.0

if scores['value'].isna().any():
    raise ValueError('Hay puntajes sin valor numérico.')
if not scores['value'].between(0, 10).all():
    raise ValueError('Los puntajes deben estar entre 0 y 10.')

# Un valor representativo por hallazgo evita contar varias fuentes CVSS
# como si fueran vulnerabilidades distintas.
representative_scores = (
    scores.sort_values(
        ['finding_index', 'value', 'source'],
        ascending=[True, False, True],
        na_position='last',
    )
    .drop_duplicates('finding_index')
    .rename(columns={
        'value': 'representative_score',
        'system': 'score_system',
        'source': 'score_source',
        'version': 'score_version',
    })
)

scored_findings = (
    findings.reset_index(names='finding_index')
    .merge(
        representative_scores[
            [
                'finding_index',
                'representative_score',
                'score_system',
                'score_source',
                'score_version',
            ]
        ],
        on='finding_index',
        how='inner',
        validate='one_to_one',
    )
)
scored_findings['high_score'] = (
    scored_findings['representative_score'] >= HIGH_SCORE_THRESHOLD
)

display(scored_findings.head())

### Resultados

La primera tabla resume la cobertura de puntuación por herramienta. La segunda compara los repositorios mediante cantidades absolutas y proporciones. `proporcion_altos` utiliza como denominador únicamente los hallazgos con puntaje; `proporcion_altos_herramienta` muestra qué fracción de todos los hallazgos altos de esa herramienta pertenece al repositorio.

In [ ]:
hallazgos_por_herramienta = findings.groupby('tool').size().rename('hallazgos')
puntuados_por_herramienta = (
    scored_findings.groupby('tool').size().rename('hallazgos_puntuados')
)
altos_por_herramienta = (
    scored_findings.groupby('tool')['high_score'].sum().rename('hallazgos_altos')
)

cobertura_puntajes = (
    pd.concat(
        [hallazgos_por_herramienta, puntuados_por_herramienta, altos_por_herramienta],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)
cobertura_puntajes['cobertura'] = (
    cobertura_puntajes['hallazgos_puntuados'] / cobertura_puntajes['hallazgos']
)
cobertura_puntajes['proporcion_altos'] = (
    cobertura_puntajes['hallazgos_altos']
    / cobertura_puntajes['hallazgos_puntuados'].replace(0, float('nan'))
)

base_repositorios = (
    findings.groupby(['repository', 'tool'])
    .size()
    .rename('hallazgos')
    .reset_index()
)
metricas_puntuadas = (
    scored_findings.groupby(['repository', 'tool'])
    .agg(
        hallazgos_puntuados=('finding_index', 'size'),
        hallazgos_altos=('high_score', 'sum'),
        puntaje_mediano=('representative_score', 'median'),
        puntaje_maximo=('representative_score', 'max'),
    )
    .reset_index()
)

metricas_pregunta_3 = (
    base_repositorios.merge(
        metricas_puntuadas,
        on=['repository', 'tool'],
        how='left',
        validate='one_to_one',
    )
    .fillna({'hallazgos_puntuados': 0, 'hallazgos_altos': 0})
)
metricas_pregunta_3[['hallazgos_puntuados', 'hallazgos_altos']] = (
    metricas_pregunta_3[['hallazgos_puntuados', 'hallazgos_altos']].astype(int)
)
metricas_pregunta_3['cobertura'] = (
    metricas_pregunta_3['hallazgos_puntuados'] / metricas_pregunta_3['hallazgos']
)
metricas_pregunta_3['proporcion_altos'] = (
    metricas_pregunta_3['hallazgos_altos']
    / metricas_pregunta_3['hallazgos_puntuados'].replace(0, float('nan'))
)
totales_altos = metricas_pregunta_3.groupby('tool')['hallazgos_altos'].transform('sum')
metricas_pregunta_3['proporcion_altos_herramienta'] = (
    metricas_pregunta_3['hallazgos_altos'] / totales_altos.replace(0, float('nan'))
)
metricas_pregunta_3 = metricas_pregunta_3.sort_values(
    ['tool', 'hallazgos_altos', 'proporcion_altos', 'repository'],
    ascending=[True, False, False, True],
).reset_index(drop=True)

display(cobertura_puntajes.round(3))
display(metricas_pregunta_3.round({
    'cobertura': 3,
    'proporcion_altos': 3,
    'proporcion_altos_herramienta': 3,
    'puntaje_mediano': 1,
    'puntaje_maximo': 1,
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for axis, tool in zip(axes, ['codeql', 'grype']):
    plot_data = metricas_pregunta_3.loc[
        (metricas_pregunta_3['tool'] == tool)
        & (metricas_pregunta_3['hallazgos_altos'] > 0)
    ].sort_values('hallazgos_altos')
    axis.barh(plot_data['repository'], plot_data['hallazgos_altos'])
    axis.set_title(f'{tool.capitalize()}: hallazgos con puntaje ≥ {HIGH_SCORE_THRESHOLD:g}')
    axis.set_xlabel('Hallazgos altos')
    axis.set_ylabel('Repositorio')
    axis.grid(axis='x', alpha=0.25)

fig.tight_layout()
plt.show()

### Interpretación

In [ ]:
observaciones = []
for tool in ['codeql', 'grype']:
    tool_metrics = metricas_pregunta_3.loc[
        (metricas_pregunta_3['tool'] == tool)
        & (metricas_pregunta_3['hallazgos_altos'] > 0)
    ]
    if tool_metrics.empty:
        observaciones.append(
            f'- **{tool.capitalize()}** no registró hallazgos puntuados sobre el umbral.'
        )
        continue

    leader = tool_metrics.iloc[0]
    observations_share = leader['proporcion_altos_herramienta']
    observaciones.append(
        f"- **{tool.capitalize()}** concentra su mayor cantidad en "
        f"**{leader['repository']}**, con {leader['hallazgos_altos']} hallazgos altos "
        f"({observations_share:.1%} del total alto de la herramienta)."
    )

display(Markdown(
    '\n'.join(observaciones)
    + '\n\nLos conteos absolutos muestran concentración operativa; las proporciones '
      'indican intensidad entre los hallazgos que sí tienen puntaje. Un repositorio '
      'puede destacar en una medida y no en la otra, por lo que ambas deben '
      'consultarse junto con la cobertura.'
))

### Limitaciones

Los puntajes de CodeQL pertenecen a las reglas; los CVSS de Grype provienen de vulnerabilidades conocidas. El máximo elegido entre varias fuentes sirve para priorizar y puede ser conservador, pero no sustituye una evaluación de riesgo. No se combinan las herramientas en una única escala ni se infieren valores ausentes. La cobertura del análisis, las versiones CVSS y el tamaño de los proyectos condicionan las conclusiones.

## Pregunta 4. ¿Qué paquetes y vulnerabilidades de Grype comparten varios repositorios?

### Método

Se filtran los hallazgos de Grype y se estudian dos agrupaciones complementarias:

1. **Vulnerabilidades compartidas:** un identificador aparece en al menos dos repositorios distintos.
2. **Paquetes compartidos:** un paquete vulnerable aparece en al menos dos repositorios distintos.

`repositorios_afectados` mide la extensión dentro de la organización. `coincidencias_totales` conserva el número de registros producidos por el Miner. Las versiones y ubicaciones se mantienen para interpretar si los repositorios comparten realmente la misma dependencia afectada.

In [ ]:
grype_findings = findings.loc[findings['tool'] == 'grype'].copy()

if grype_findings.empty:
    raise ValueError('El dataset no contiene hallazgos de Grype.')

vulnerabilidades_grype = (
    grype_findings.groupby('vulnerability_id')
    .agg(
        repositorios_afectados=('repository', 'nunique'),
        coincidencias_totales=('repository', 'size'),
        paquetes_afectados=('package', 'nunique'),
        severidades=('severity', lambda values: sorted(values.dropna().unique())),
    )
    .reset_index()
)

vulnerabilidades_compartidas = (
    vulnerabilidades_grype.loc[
        vulnerabilidades_grype['repositorios_afectados'] >= 2
    ]
    .sort_values(
        ['repositorios_afectados', 'coincidencias_totales', 'vulnerability_id'],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

detalle_vulnerabilidades_compartidas = (
    grype_findings.loc[
        grype_findings['vulnerability_id'].isin(
            vulnerabilidades_compartidas['vulnerability_id']
        ),
        ['vulnerability_id', 'repository', 'package', 'version', 'severity', 'locations'],
    ]
    .sort_values(['vulnerability_id', 'repository', 'package', 'version'])
    .reset_index(drop=True)
)

display(vulnerabilidades_compartidas)
display(detalle_vulnerabilidades_compartidas.head(20))

In [ ]:
paquetes_grype = (
    grype_findings.dropna(subset=['package'])
    .groupby(['package', 'package_type'], dropna=False)
    .agg(
        repositorios_afectados=('repository', 'nunique'),
        vulnerabilidades_distintas=('vulnerability_id', 'nunique'),
        coincidencias_totales=('repository', 'size'),
        versiones_observadas=('version', lambda values: sorted(values.dropna().unique())),
    )
    .reset_index()
)

paquetes_compartidos = (
    paquetes_grype.loc[paquetes_grype['repositorios_afectados'] >= 2]
    .sort_values(
        [
            'repositorios_afectados',
            'vulnerabilidades_distintas',
            'coincidencias_totales',
            'package',
        ],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)

detalle_paquetes_compartidos = (
    grype_findings.loc[
        grype_findings['package'].isin(paquetes_compartidos['package']),
        ['package', 'package_type', 'repository', 'version', 'vulnerability_id', 'severity'],
    ]
    .sort_values(['package', 'repository', 'version', 'vulnerability_id'])
    .reset_index(drop=True)
)

display(paquetes_compartidos)
display(detalle_paquetes_compartidos.head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

top_vulnerabilidades = vulnerabilidades_compartidas.head(15).sort_values(
    'repositorios_afectados'
)
axes[0].barh(
    top_vulnerabilidades['vulnerability_id'],
    top_vulnerabilidades['repositorios_afectados'],
)
axes[0].set_title('Vulnerabilidades compartidas')
axes[0].set_xlabel('Repositorios afectados')
axes[0].grid(axis='x', alpha=0.25)

top_paquetes = paquetes_compartidos.head(15).sort_values('repositorios_afectados')
axes[1].barh(
    top_paquetes['package'],
    top_paquetes['repositorios_afectados'],
)
axes[1].set_title('Paquetes vulnerables compartidos')
axes[1].set_xlabel('Repositorios afectados')
axes[1].grid(axis='x', alpha=0.25)

fig.tight_layout()
plt.show()

### Interpretación

In [ ]:
observaciones_pregunta_4 = [
    f'- Se encontraron **{len(vulnerabilidades_compartidas)} vulnerabilidades** '
    'presentes en al menos dos repositorios.',
    f'- Se encontraron **{len(paquetes_compartidos)} paquetes vulnerables** '
    'presentes en al menos dos repositorios.',
]

if not vulnerabilidades_compartidas.empty:
    lider_vulnerabilidad = vulnerabilidades_compartidas.iloc[0]
    observaciones_pregunta_4.append(
        f"- La vulnerabilidad más extendida es "
        f"**{lider_vulnerabilidad['vulnerability_id']}**, presente en "
        f"**{lider_vulnerabilidad['repositorios_afectados']} repositorios**."
    )

if not paquetes_compartidos.empty:
    lider_paquete = paquetes_compartidos.iloc[0]
    observaciones_pregunta_4.append(
        f"- El paquete vulnerable más extendido es **{lider_paquete['package']}**, "
        f"presente en **{lider_paquete['repositorios_afectados']} repositorios** "
        f"y relacionado con **{lider_paquete['vulnerabilidades_distintas']} "
        "identificadores distintos**."
    )

display(Markdown(
    '\n'.join(observaciones_pregunta_4)
    + '\n\nLa repetición sugiere dependencias comunes que pueden revisarse de manera '
      'coordinada. Antes de priorizar una actualización debe consultarse el detalle '
      'de versiones, porque compartir el nombre del paquete no implica usar la misma '
      'versión ni tener la misma exposición.'
))

### Limitaciones

Grype analiza componentes declarados en los SBOM y los contrasta con bases de vulnerabilidades conocidas. Una coincidencia no demuestra que el código vulnerable sea alcanzable durante la ejecución. Un mismo paquete puede aparecer con versiones distintas, y un identificador puede relacionarse con más de una fuente o paquete. Los resultados dependen de la completitud del SBOM y del estado de la base de datos de Grype al momento del análisis.

## Pregunta 5. ¿Existe relación entre los conteos de CodeQL y Grype?

### Introducción

Esta pregunta estudia si los repositorios con más hallazgos de código también tienden a presentar más vulnerabilidades conocidas en sus dependencias. Solo se incluyen repositorios cuyo estado sea `analyzed` tanto en CodeQL como en Grype; así, un análisis fallido no se interpreta como un conteo igual a cero.

### Método: correlación de Spearman

La correlación de Spearman mide si dos variables mantienen una relación **monotónica**: cuando una aumenta, la otra tiende a aumentar o disminuir, aunque la relación no sea lineal. En lugar de comparar directamente los conteos, el método realiza estos pasos:

1. Ordena los repositorios según sus hallazgos de CodeQL y asigna un rango a cada uno.
2. Repite el proceso con los hallazgos de Grype.
3. Cuando varios repositorios empatan, les asigna el rango promedio de las posiciones que ocupan.
4. Calcula la correlación de Pearson entre ambas columnas de rangos.

El coeficiente resultante, **rho de Spearman (`ρ`)**, está entre `-1` y `1`:

- `ρ` cercano a `1`: los repositorios con más hallazgos de CodeQL tienden a tener más hallazgos de Grype.
- `ρ` cercano a `-1`: los repositorios con más hallazgos de una herramienta tienden a tener menos de la otra.
- `ρ` cercano a `0`: no se observa una relación monotónica clara.

Se usa Spearman porque los conteos tienen muchos empates y valores extremos, y no se puede asumir una distribución normal ni una relación lineal. El cálculo se realiza directamente con los rangos para no incorporar una dependencia adicional.

In [ ]:
repositorios_cobertura_completa = repositories.loc[
    (repositories['codeql_status'] == 'analyzed')
    & (repositories['grype_status'] == 'analyzed'),
    ['repository', 'codeql_status', 'grype_status'],
]

conteos_relacion = (
    conteos_por_repositorio.reset_index()
    .merge(
        repositorios_cobertura_completa,
        on='repository',
        how='inner',
        validate='one_to_one',
    )
    .rename(columns={'codeql': 'hallazgos_codeql', 'grype': 'hallazgos_grype'})
)

if len(conteos_relacion) < 2:
    raise ValueError('Se necesitan al menos dos repositorios con cobertura completa.')
if (
    conteos_relacion['hallazgos_codeql'].nunique() < 2
    or conteos_relacion['hallazgos_grype'].nunique() < 2
):
    raise ValueError('Los conteos no tienen variación suficiente para calcular Spearman.')

conteos_relacion['rango_codeql'] = conteos_relacion['hallazgos_codeql'].rank(
    method='average'
)
conteos_relacion['rango_grype'] = conteos_relacion['hallazgos_grype'].rank(
    method='average'
)
rho_spearman = conteos_relacion['rango_codeql'].corr(
    conteos_relacion['rango_grype']
)

conteos_relacion = conteos_relacion.sort_values(
    ['hallazgos_codeql', 'hallazgos_grype', 'repository'],
    ascending=[False, False, True],
).reset_index(drop=True)

print(f'Repositorios incluidos: {len(conteos_relacion)}')
print(f'Rho de Spearman: {rho_spearman:.3f}')
display(conteos_relacion)

In [ ]:
fig, axis = plt.subplots(figsize=(10, 7))
axis.scatter(
    conteos_relacion['hallazgos_codeql'],
    conteos_relacion['hallazgos_grype'],
    alpha=0.8,
)

for row in conteos_relacion.itertuples():
    axis.annotate(
        row.repository.removeprefix(f"{dataset['organization']}/"),
        (row.hallazgos_codeql, row.hallazgos_grype),
        xytext=(4, 4),
        textcoords='offset points',
        fontsize=8,
    )

axis.set_title(f'Conteos por repositorio: Spearman ρ = {rho_spearman:.3f}')
axis.set_xlabel('Hallazgos de CodeQL')
axis.set_ylabel('Hallazgos de Grype')
axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### Interpretación

In [ ]:
magnitud = abs(rho_spearman)
if magnitud >= 0.7:
    intensidad = 'fuerte'
elif magnitud >= 0.4:
    intensidad = 'moderada'
elif magnitud >= 0.2:
    intensidad = 'débil'
else:
    intensidad = 'muy débil'

if rho_spearman > 0:
    direccion = 'positiva'
elif rho_spearman < 0:
    direccion = 'negativa'
else:
    direccion = 'nula'

display(Markdown(
    f'Entre los **{len(conteos_relacion)} repositorios** con cobertura completa, '
    f'el coeficiente es **ρ = {rho_spearman:.3f}**, lo que describe una relación '
    f'**{direccion} y {intensidad}** entre los rangos de hallazgos de CodeQL y Grype. '
    'Esta clasificación resume la magnitud observada y no constituye por sí sola '
    'una prueba de significancia estadística.'
))

### Limitaciones

La correlación describe asociación, no causalidad: los hallazgos de código no producen necesariamente vulnerabilidades en dependencias. Los conteos no están normalizados por líneas de código, cantidad de dependencias ni tamaño del repositorio. La muestra es pequeña y contiene empates y valores extremos; por ello se informan el coeficiente, el gráfico y el número de repositorios incluidos, sin afirmar significancia estadística. CodeQL y Grype además observan dimensiones de seguridad diferentes.

## Pregunta 6. ¿En qué archivos se concentran los hallazgos de CodeQL dentro de cada repositorio?

### Método

Se estudian los hallazgos de CodeQL y sus rutas en `locations`. Cada registro del dataset recibe un `finding_index`; si menciona varias veces el mismo archivo, se cuenta una sola vez en ese archivo. Si menciona archivos distintos, contribuye una vez a cada uno.

Se presentan los tres archivos con más hallazgos por repositorio, el número de reglas distintas y la proporción respecto de todos los hallazgos de CodeQL del repositorio. Los empates se resuelven por ruta alfabética para obtener un máximo de tres archivos reproducible.

La concentración conjunta del top 3 usa la **unión de hallazgos** asociados a esos archivos: un hallazgo presente en dos archivos del top se cuenta una sola vez. Las proporciones individuales por archivo no deben sumarse.

Se informa además la cobertura de ubicación y el estado de CodeQL para todos los repositorios. Los repositorios sin hallazgos tienen proporciones no definidas; un fallo de análisis no se interpreta como ausencia de problemas.


In [ ]:
codeql_archivos = (
    findings.loc[findings['tool'] == 'codeql']
    .reset_index(names='finding_index')
    .copy()
)

def rutas_unicas(locations):
    if not isinstance(locations, list):
        return []
    return sorted({
        location.strip().replace('\\', '/')
        for location in locations
        if isinstance(location, str) and location.strip()
    })

codeql_archivos['archivos'] = codeql_archivos['locations'].map(rutas_unicas)
ubicaciones_codeql = (
    codeql_archivos[['finding_index', 'repository', 'vulnerability_id', 'archivos']]
    .explode('archivos')
    .rename(columns={'archivos': 'archivo'})
    .dropna(subset=['archivo'])
    .drop_duplicates(['finding_index', 'archivo'])
)

resumen_archivos = (
    repositories[['repository', 'codeql_status']]
    .merge(
        codeql_archivos.groupby('repository').size().rename('hallazgos_codeql'),
        on='repository', how='left', validate='one_to_one',
    )
    .merge(
        ubicaciones_codeql.groupby('repository').agg(
            hallazgos_con_ubicacion=('finding_index', 'nunique'),
            archivos_con_hallazgos=('archivo', 'nunique'),
        ),
        on='repository', how='left', validate='one_to_one',
    )
)
columnas_conteo = ['hallazgos_codeql', 'hallazgos_con_ubicacion', 'archivos_con_hallazgos']
resumen_archivos[columnas_conteo] = resumen_archivos[columnas_conteo].fillna(0).astype(int)
resumen_archivos['hallazgos_sin_ubicacion'] = (
    resumen_archivos['hallazgos_codeql'] - resumen_archivos['hallazgos_con_ubicacion']
)
resumen_archivos['cobertura_ubicacion'] = (
    resumen_archivos['hallazgos_con_ubicacion']
    / resumen_archivos['hallazgos_codeql'].replace(0, float('nan'))
)

hallazgos_por_archivo = (
    ubicaciones_codeql.groupby(['repository', 'archivo'])
    .agg(
        hallazgos=('finding_index', 'nunique'),
        reglas_distintas=('vulnerability_id', 'nunique'),
    )
    .reset_index()
    .merge(
        resumen_archivos[['repository', 'codeql_status', 'hallazgos_codeql']],
        on='repository', how='left', validate='many_to_one',
    )
)
hallazgos_por_archivo['proporcion_del_repositorio'] = (
    hallazgos_por_archivo['hallazgos'] / hallazgos_por_archivo['hallazgos_codeql']
)
hallazgos_por_archivo = hallazgos_por_archivo.sort_values(
    ['repository', 'hallazgos', 'archivo'], ascending=[True, False, True],
).reset_index(drop=True)
top_archivos = hallazgos_por_archivo.groupby('repository', sort=False).head(3).copy()

hallazgos_top_3 = (
    ubicaciones_codeql.merge(
        top_archivos[['repository', 'archivo']],
        on=['repository', 'archivo'], how='inner', validate='many_to_one',
    )
    .groupby('repository')['finding_index'].nunique()
    .rename('hallazgos_en_top_3')
)
resumen_archivos = resumen_archivos.merge(
    hallazgos_top_3, on='repository', how='left', validate='one_to_one',
)
resumen_archivos['hallazgos_en_top_3'] = resumen_archivos['hallazgos_en_top_3'].fillna(0).astype(int)
resumen_archivos['proporcion_top_3'] = (
    resumen_archivos['hallazgos_en_top_3']
    / resumen_archivos['hallazgos_codeql'].replace(0, float('nan'))
)
resumen_archivos = resumen_archivos.sort_values(
    ['hallazgos_codeql', 'repository'], ascending=[False, True],
).reset_index(drop=True)

display(resumen_archivos.round(3))
display(top_archivos.round({'proporcion_del_repositorio': 3}))


### Resultados

Cada panel muestra los archivos principales de un repositorio. Las etiquetas indican el conteo y su porcentaje respecto de los hallazgos de CodeQL de ese repositorio. Las escalas horizontales son independientes; la tabla anterior permite comparar cantidades exactas y consultar la cobertura.


In [ ]:
repositorios_con_archivos = top_archivos['repository'].drop_duplicates().tolist()
if not repositorios_con_archivos:
    print('No hay hallazgos de CodeQL con rutas de archivo disponibles.')
else:
    filas = (len(repositorios_con_archivos) + 1) // 2
    fig, axes = plt.subplots(filas, 2, figsize=(18, 3.5 * filas), squeeze=False)
    for axis, repository in zip(axes.flat, repositorios_con_archivos):
        datos_archivo = top_archivos.loc[
            top_archivos['repository'] == repository
        ].sort_values(['hallazgos', 'archivo'])
        barras = axis.barh(datos_archivo['archivo'], datos_archivo['hallazgos'])
        axis.bar_label(barras, labels=[
            f'{row.hallazgos} ({row.proporcion_del_repositorio:.1%})'
            for row in datos_archivo.itertuples()
        ], padding=3)
        axis.set_xlim(0, datos_archivo['hallazgos'].max() * 1.5)
        axis.set_title(repository)
        axis.set_xlabel('Hallazgos de CodeQL')
        axis.tick_params(axis='y', labelsize=8)
        axis.grid(axis='x', alpha=0.25)
    for axis in list(axes.flat)[len(repositorios_con_archivos):]:
        axis.set_visible(False)
    fig.tight_layout()
    plt.show()


### Interpretación


In [ ]:
observaciones_archivos = []
for row in resumen_archivos.itertuples():
    if row.hallazgos_codeql == 0:
        continue
    if row.hallazgos_con_ubicacion == 0:
        observaciones_archivos.append(
            f'- **{row.repository}**: {row.hallazgos_codeql} hallazgos sin ruta; '
            'no se puede calcular su concentración por archivo.'
        )
        continue
    cantidad_archivos = int((top_archivos['repository'] == row.repository).sum())
    observaciones_archivos.append(
        f'- **{row.repository}**: los {cantidad_archivos} archivos principales '
        f'reúnen {row.hallazgos_en_top_3} de {row.hallazgos_codeql} hallazgos '
        f'({row.proporcion_top_3:.1%}); cobertura de ubicación '
        f'{row.cobertura_ubicacion:.1%}, estado `{row.codeql_status}`.'
    )

sin_analisis_completo = int((resumen_archivos['codeql_status'] != 'analyzed').sum())
observaciones_archivos.append(
    f'- **{sin_analisis_completo} repositorios** tienen un estado de CodeQL '
    'distinto de `analyzed`; sus conteos no representan una revisión completa.'
)
display(Markdown('\n'.join(observaciones_archivos)))


### Limitaciones

Se cuentan alertas de CodeQL, no vulnerabilidades confirmadas. Las rutas proceden de las ubicaciones conservadas por el Miner; no representan necesariamente todo el recorrido del flujo de datos de una alerta.

La concentración no está normalizada por líneas de código ni tamaño del archivo. Un porcentaje alto puede corresponder a muy pocos hallazgos o a un repositorio pequeño. Se incluyen archivos de pruebas y ejemplos: su contexto debe revisarse antes de priorizar. Los análisis fallidos o parciales y las ubicaciones ausentes limitan las conclusiones.


## Resultados para el Visualizer
Exportar aquí, al desarrollar los análisis, las métricas y tablas derivadas junto con sus definiciones, filtros y observaciones. Mantener la evidencia original en el Miner.